In [1]:
import datetime
from openpyxl import load_workbook
from openpyxl.styles import NamedStyle
from openpyxl.utils.cell import get_column_letter

# Define file paths
agent_file = 'New Microsoft Excel Worksheet.xlsx'  # Replace with the path to the agent's file
my_file = 'Test (1).xlsx'        # Replace with the path to your file

# Load the workbooks
agent_wb = load_workbook(agent_file, data_only=True)  # `data_only` preserves values as they appear in the cells
my_wb = load_workbook(my_file)

# Get available sheet names from the agent's workbook
agent_sheets = agent_wb.sheetnames
my_sheets = my_wb.sheetnames

# Display available sheets and ask the user to select one
print("Available sheets in the agent's workbook:")
for idx, sheet_name in enumerate(agent_sheets, 1):
    print(f"{idx}. {sheet_name}")

# Get the user's selection
while True:
    try:
        choice = int(input("Enter the number of the sheet you want to select: "))
        if 1 <= choice <= len(agent_sheets):
            selected_sheet = agent_sheets[choice - 1]
            break
        else:
            print("Invalid selection. Please select a valid sheet number.")
    except ValueError:
        print("Invalid input. Please enter a number.")

# Check if the selected sheet exists in my workbook
if selected_sheet in my_sheets:
    print(f"Sheet '{selected_sheet}' already exists in your workbook.")
    my_ws = my_wb[selected_sheet]
else:
    print(f"Sheet '{selected_sheet}' does not exist in your workbook. Creating a new sheet...")
    # Create a new sheet with the same name
    my_ws = my_wb.create_sheet(title=selected_sheet)

    # Copy headers from the previous month's sheet
    if len(my_sheets) > 0:
        # Assuming the last sheet in `my_wb` is the previous month's sheet
        previous_month_ws = my_wb[my_sheets[-1]]
        for col in range(1, previous_month_ws.max_column + 1):
            my_ws.cell(row=1, column=col).value = previous_month_ws.cell(row=1, column=col).value
        print(f"Headers copied from the previous month's sheet '{my_sheets[-1]}' to the new sheet.")
    else:
        print("No previous sheet found in your workbook to copy headers from. New sheet will be empty.")


# Select the corresponding sheet in the agent's workbook
agent_ws = agent_wb[selected_sheet]

# Identify the last non-empty row in `my_sheet.xlsx`
last_row_my_sheet = my_ws.max_row
while last_row_my_sheet > 1 and not any(my_ws.cell(row=last_row_my_sheet, column=col).value for col in range(1, my_ws.max_column + 1)):
    last_row_my_sheet -= 1  # Move up if the row is empty

# Get the SO number of the last row in your sheet
last_so_number = my_ws.cell(row=last_row_my_sheet, column=1).value

# Find the row with the last SO number in the agent sheet
agent_last_row = None
for row in range(1, agent_ws.max_row + 1):
    if agent_ws.cell(row=row, column=1).value == last_so_number:
        agent_last_row = row
        break

# Start appending from the row after the identified last SO number
start_row = (agent_last_row + 1) if agent_last_row else 2  # Skip headers if appending all rows

# Collect rows to append
rows_to_append = []
for row in range(start_row, agent_ws.max_row + 1):
    if agent_ws.cell(row=row, column=1).value:  # Ensure the first column is not empty
        rows_to_append.append([agent_ws.cell(row=row, column=col).value for col in range(1, agent_ws.max_column + 1)])

# Define a short date style
short_date_style = NamedStyle(name="short_date", number_format="MM/DD/YYYY")

# Add the style to the workbook if not already present
if "short_date" not in my_wb.named_styles:
    my_wb.add_named_style(short_date_style)

# Append the rows with proper date handling
for i, row_data in enumerate(rows_to_append, start=last_row_my_sheet + 1):
    for col, value in enumerate(row_data, start=1):
        cell = my_ws.cell(row=i, column=col, value=value)

        # Check if the value is a datetime object
        #if isinstance(value, (datetime.date, datetime.datetime)):
         #   cell.style = short_date_style  # Apply short date format

# Save the workbook3
my_wb.save(my_file)
print(f"{len(rows_to_append)} new rows appended to {my_file}.")


Available sheets in the agent's workbook:
1. DEC 2024
2. JAN 2025
3. FEB 2025
Sheet 'FEB 2025' already exists in your workbook.
0 new rows appended to Test (1).xlsx.


In [ ]:
import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Plant-specific mappings for selector adjustments
plant_mapping = {
    "bodyline": {
        "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9",
        "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1",
        "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }
}

# Function to get initial setup values
def get_initial_setup():
    print("Please provide the following initial setup values:")
    
    # Get SAPLMEGUI initial value
    sap_gui_value = input("Enter the initial value for SAPLMEGUI (e.g., 0013): ").strip()
    
    # Get rates and weights
    rates_weights = {
        "hong_kong_exw_rate": float(input("Enter the rate for Hong Kong EXW shipments: ")),
        "hong_kong_fob_rate": float(input("Enter the rate for Hong Kong FOB shipments: ")),
        "china_exw_rate": float(input("Enter the rate for China EXW shipments: ")),
        "china_fob_rate": float(input("Enter the rate for China FOB shipments: ")),
        "max_volume_hong_kong": float(input("Enter the maximum volume for Hong Kong shipments: ")),
        "max_volume_china": float(input("Enter the maximum volume for China shipments: "))
    }
    
    return sap_gui_value, rates_weights

# Function to replace SAPLMEGUI:XXXX with the user-provided value
def update_sap_gui_ids(sap_gui_value, plant_mapping):
    for plant, mappings in plant_mapping.items():
        for key, value in mappings.items():
            mappings[key] = value.replace("SAPLMEGUI:0013", f"SAPLMEGUI:{sap_gui_value}")
    return plant_mapping

# SAP Login Function
def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

# Function to process valid POs with SAP
def process_valid_pos_with_sap(df, session, plant_mapping, rates_weights):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, plant_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Extract PO details from SAP
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Plant-specific handling
                if plant_name in plant_mapping:
                    session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                    created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                    session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                    inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
                else:
                    session.findById(default_mapping["tab_selector9"]).select()
                    created_by = session.findById(default_mapping["created_by"]).text
                    session.findById(default_mapping["tab_selector1"]).select()
                    inco_term = session.findById(default_mapping["inco_term"]).text
                                
                    # Default handling
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9"
                    ).select()
                    created_by = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
                    ).text
                    
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1"
                    ).select()
                    inco_term = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                    ).text
                
                # Add extracted values to the sets
                merchant.add(created_by)
                incoterm.add(inco_term)
                
            except Exception as e:
                print(f"Error processing PO {purchase_order_number}: {str(e)}")
                continue

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[29]='Manual Check'
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
        # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[29]='Manual Check'
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs) # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
            
            # Set status
            row[28] = "Financial Approval(Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > rates_weights["max_volume_hong_kong"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > rates_weights["max_volume_china"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Received approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery term confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > rates_weights["max_volume_hong_kong"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"Hong Kong shipment, volume > {rates_weights['max_volume_hong_kong']}, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[29] = "Received approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > rates_weights["max_volume_china"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["china_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"China shipment, volume > {rates_weights['max_volume_china']}, Financial approval set to 'APP'.")
                    
                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received approval"  # Mail status
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm match found. Status: 'All Good'. Mail to Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

        processed_rows.append(row)

    # Create a new DataFrame with only the relevant columns
    processed_df = pd.DataFrame(processed_rows, columns=df.columns)
    return processed_df

# Main Execution
if __name__ == "__main__":
    # Get initial setup values
    sap_gui_value, rates_weights = get_initial_setup()
    
    # Update SAP GUI IDs with the user-provided value
    plant_mapping = update_sap_gui_ids(sap_gui_value, plant_mapping)
    
    # Load the Excel file and filter rows
  # Replace with your sheet name
    df = pd.read_excel(my_file, sheet_name=selected_sheet)
    
    # Filter out rows where the "status" column is blank
    df_filtered = df[df['status'].isna()]
    
    # Login to SAP
    session = sap_login(username='maheswaranl', password='Srilanka@2024', connection_name='PDM', transaction_code='me23n')
    
    # Process the filtered DataFrame
    df_with_merchant_and_terms = process_valid_pos_with_sap(df_filtered, session, plant_mapping, rates_weights)
    
    # Display the relevant columns
    relevant_columns = [0, 3, 5, 6, 7, 12, 13, 14, 17, 18, 28, 29, 30]  # SO, PO #, Merchant, Terms, Remarks
    display(df_with_merchant_and_terms.iloc[:, relevant_columns])

Please provide the following initial setup values:
Logging in...
Login successful.
Navigating to transaction me23n...
Transaction opened successfully.
POs BEMIS contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
POs CHARGEURS contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
POs BEST PACIFIC  contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
Hong Kong shipment, volume > 15.0, Financial approval set to 'APP'.
Hong Kong shipment, volume <= 15.0, Incoterm mismatch. Status: 'Del

,S/O,Consignee,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks2
0,811604,BODYLINE,"4502357146, 4502359043, 4502359049, 4502359077...","Ashank, VidyaH",EXW,APP,87.740,32.8990,HONG KONG,530.0,Financial Approval(Bodyline),NaN,NaN
1,811606,UNICHELA-1900,4900143441,AVISHKAB,EXW,Cstd,74.340,21.7100,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
2,811607,BODYLINE,"4502359056, 4502365299, 4502365296, 4502365300...",VidyaH,EXW,APP,11.161,8.1830,HONG KONG,70.0,Financial Approval(Bodyline),NaN,NaN
3,811608,BODYLINE,4502333403,ISURUSIR,EXW,APP,29.000,8.3500,HONG KONG,170.0,Financial Approval(Bodyline),NaN,NaN
4,811609,BODYLINE,4502372521,ISURUSIR,EXW,APP,24.000,13.3600,HONG KONG,140.0,Financial Approval(Bodyline),NaN,NaN
5,811610,MAS CAPITAL,4900145360,MANOJVID,EXW,Cstd,28.110,22.2110,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
6,811612,MAS CAPITAL,4900143920,KAVISHKARA,EXW,Cstd,17.000,45.0900,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
7,811826,UNICHELA-A053,4800474691,YasaraJ,EXW,APP,140.740,91.8500,HONG KONG,840.0,Approval,NaN,NaN
8,811827,SILUETA,4400043673,N/A,N/A,CE,14.700,5.0100,HONG KONG,NaN,Delivery term confirmation,Manual Check,Extraction Skipped for Silueta
9,811613,BODYLINE,"4502363384, 4502364384",DANUSHIW_C,EXW,APP,84.800,23.3800,HONG KONG,510.0,Financial Approval(Bodyline),NaN,NaN


In [7]:
import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Plant-specific mappings for selector adjustments
plant_mapping = {
    "bodyline": {
        "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9",
        "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1",
        "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }
}

# Function to get initial setup values
def get_initial_setup():
    print("Please provide the following initial setup values:")
    
    # Get SAPLMEGUI initial value
    sap_gui_value = input("Enter the initial value for SAPLMEGUI (e.g., 0013): ").strip()
    
    # Get rates and weights
    rates_weights = {
        "hong_kong_exw_rate": float(input("Enter the rate for Hong Kong EXW shipments: ")),
        "hong_kong_fob_rate": float(input("Enter the rate for Hong Kong FOB shipments: ")),
        "china_exw_rate": float(input("Enter the rate for China EXW shipments: ")),
        "china_fob_rate": float(input("Enter the rate for China FOB shipments: ")),
        "max_volume_hong_kong": float(input("Enter the maximum volume for Hong Kong shipments: ")),
        "max_volume_china": float(input("Enter the maximum volume for China shipments: "))
    }
    
    return sap_gui_value, rates_weights

# Function to replace SAPLMEGUI:XXXX with the user-provided value
def update_sap_gui_ids(sap_gui_value, plant_mapping):
    for plant, mappings in plant_mapping.items():
        for key, value in mappings.items():
            mappings[key] = value.replace("SAPLMEGUI:0013", f"SAPLMEGUI:{sap_gui_value}")
    return plant_mapping

# SAP Login Function
def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

# Function to process valid POs with SAP
def process_valid_pos_with_sap(df, session, plant_mapping, rates_weights):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, plant_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Extract PO details from SAP
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Plant-specific handling
                if plant_name in plant_mapping:
                    session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                    created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                    session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                    inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
                else:
                    session.findById(default_mapping["tab_selector9"]).select()
                    created_by = session.findById(default_mapping["created_by"]).text
                    session.findById(default_mapping["tab_selector1"]).select()
                    inco_term = session.findById(default_mapping["inco_term"]).text
                                
                    # Default handling
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9"
                    ).select()
                    created_by = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
                    ).text
                    
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1"
                    ).select()
                    inco_term = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                    ).text
                
                # Add extracted values to the sets
                merchant.add(created_by)
                incoterm.add(inco_term)
                
            except Exception as e:
                print(f"Error processing PO {purchase_order_number}: {str(e)}")
                continue

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[29]='Manual Check'
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
        # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[29]='Manual Check'
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs) # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
            
            # Set status
            row[28] = "Financial Approval(Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > rates_weights["max_volume_hong_kong"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > rates_weights["max_volume_china"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Received approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery term confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > rates_weights["max_volume_hong_kong"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"Hong Kong shipment, volume > {rates_weights['max_volume_hong_kong']}, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[29] = "Received approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > rates_weights["max_volume_china"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["china_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"China shipment, volume > {rates_weights['max_volume_china']}, Financial approval set to 'APP'.")
                    
                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received approval"  # Mail status
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm match found. Status: 'All Good'. Mail to Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

        processed_rows.append(row)

    # Create a new DataFrame with only the relevant columns
    processed_df = pd.DataFrame(processed_rows, columns=df.columns)
    return processed_df


In [ ]:
import json
import os
import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


# SAP Login Function
def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

# Function to process valid POs with SAP
def process_valid_pos_with_sap(df, session, plant_mapping,default_mapping, rates_weights):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, plant_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Extract PO details from SAP
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Plant-specific handling
                if plant_name in plant_mapping:
                    session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                    created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                    session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                    inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
                else:
                    session.findById(default_mapping["tab_selector9"]).select()
                    created_by = session.findById(default_mapping["created_by"]).text
                    session.findById(default_mapping["tab_selector1"]).select()
                    inco_term = session.findById(default_mapping["inco_term"]).text
                                
                    
                # Add extracted values to the sets
                merchant.add(created_by)
                incoterm.add(inco_term)
                
            except Exception as e:
                print(f"Error processing PO {purchase_order_number}: {str(e)}")
                continue

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[29]='Manual Check'
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
        # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[29]='Manual Check'
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs) # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
            
            # Set status
            row[28] = "Financial Approval(Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > rates_weights["max_volume_hong_kong"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > rates_weights["max_volume_china"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Received approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery term confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > rates_weights["max_volume_hong_kong"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"Hong Kong shipment, volume > {rates_weights['max_volume_hong_kong']}, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[29] = "Received approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > rates_weights["max_volume_china"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["china_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"China shipment, volume > {rates_weights['max_volume_china']}, Financial approval set to 'APP'.")
                    
                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received approval"  # Mail status
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm match found. Status: 'All Good'. Mail to Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

        processed_rows.append(row)

    # Create a new DataFrame with only the relevant columns
    processed_df = pd.DataFrame(processed_rows, columns=df.columns)
    return processed_df

def get_updated_plant_mapping(sap_gui_value):
    return {
        "bodyline": {
            "tab_selector9": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                             "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                             "tabpTABHDT9",
            "created_by": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                          "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                          "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
            "tab_selector1": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                             "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                             "tabpTABHDT1",
            "inco_term": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
        }
    }

# Function to update placeholders in default (else block)
def get_updated_default_mapping(default_sap_gui_value):
    return {
        "tab_selector9": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{default_sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT9",
        "created_by": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{default_sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{default_sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT1",
        "inco_term": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{default_sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }

# Function to load configuration from a JSON file
def load_config(config_file='config.json'):
    if os.path.exists(config_file):
        with open(config_file, 'r') as file:
            return json.load(file)
    return None

# Function to save configuration to a JSON file
def save_config(config, config_file='config.json'):
    with open(config_file, 'w') as file:
        json.dump(config, file, indent=4)

# Function to get initial setup values
def get_initial_setup(config):
    if config:
        print("Current configuration:")
        print(json.dumps(config, indent=4))
        change = input("Do you want to change the configuration? (yes/no): ").strip().lower()
        if change != 'yes':
            return config['sap_gui_value'], config['rates_weights'], config['default_sap_gui_value']
    
    print("Please provide the following initial setup values:")
    
    # Get SAPLMEGUI initial value
    sap_gui_value = input("Enter SAPLMEGUI value for 'bodyline' plant mapping (e.g., 0013): ").strip()
    default_sap_gui_value = input("Enter SAPLMEGUI value for default (else block) handling (e.g., 0013): ").strip()
        
    # Get rates and weights
    rates_weights = {
        "hong_kong_exw_rate": float(input("Enter the rate for Hong Kong EXW shipments: ")),
        "hong_kong_fob_rate": float(input("Enter the rate for Hong Kong FOB shipments: ")),
        "china_exw_rate": float(input("Enter the rate for China EXW shipments: ")),
        "china_fob_rate": float(input("Enter the rate for China FOB shipments: ")),
        "max_volume_hong_kong": float(input("Enter the maximum volume for Hong Kong shipments: ")),
        "max_volume_china": float(input("Enter the maximum volume for China shipments: "))
    }
    

    
    # Save the new configuration
    new_config = {
        "sap_gui_value": sap_gui_value,
        "rates_weights": rates_weights,
        "default_sap_gui_value":default_sap_gui_value
    }
    save_config(new_config)
    
    return sap_gui_value, rates_weights,default_sap_gui_value 

# Main Execution
if __name__ == "__main__":
    # Load existing configuration
    config = load_config()
    
    # Get initial setup values
    sap_gui_value, rates_weights, default_sap_gui_value = get_initial_setup(config)
    
    # Update SAP GUI IDs with the user-provided value
    plant_mapping = get_updated_plant_mapping(sap_gui_value)
    default_mapping = get_updated_default_mapping(default_sap_gui_value)

    # Load the Excel file and filter rows
    df = pd.read_excel(my_file, sheet_name=selected_sheet)
    
    # Filter out rows where the "status" column is blank
    df_filtered = df[df['status'].isna()]
    
    # Login to SAP
    session = sap_login(username='maheswaranl', password='Srilanka@2024', connection_name='PDM', transaction_code='me23n')
    
    # Process the filtered DataFrame
    df_with_merchant_and_terms = process_valid_pos_with_sap(df_filtered, session, plant_mapping, default_mapping,rates_weights)
    
    # Display the relevant columns
    relevant_columns = [0, 3, 5, 6, 7, 12, 13, 14, 17, 18, 28, 29, 30]  # SO, PO #, Merchant, Terms, Remarks
    display(df_with_merchant_and_terms.iloc[:, relevant_columns])

Current configuration:
{
    "sap_gui_value": "0010",
    "rates_weights": {
        "hong_kong_exw_rate": 6.0,
        "hong_kong_fob_rate": 5.5,
        "china_exw_rate": 7.0,
        "china_fob_rate": 6.2,
        "max_volume_hong_kong": 15.0,
        "max_volume_china": 70.0
    },
    "default_sap_gui_value": "0013"
}


KeyError: 'plant_mapping'

In [5]:
df_with_merchant_and_terms.iloc[:, relevant_columns]

,S/O,Consignee,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks2
0,811604,BODYLINE,"4502357146, 4502359043, 4502359049, 4502359077...","VidyaH, Ashank",EXW,APP,87.740,32.8990,HONG KONG,530.0,Financial Approval(Bodyline),NaN,NaN
1,811606,UNICHELA-1900,4900143441,AVISHKAB,EXW,Cstd,74.340,21.7100,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
2,811607,BODYLINE,"4502359056, 4502365299, 4502365296, 4502365300...",VidyaH,EXW,APP,11.161,8.1830,HONG KONG,70.0,Financial Approval(Bodyline),NaN,NaN
3,811608,BODYLINE,4502333403,ISURUSIR,EXW,APP,29.000,8.3500,HONG KONG,170.0,Financial Approval(Bodyline),NaN,NaN
4,811609,BODYLINE,4502372521,ISURUSIR,EXW,APP,24.000,13.3600,HONG KONG,140.0,Financial Approval(Bodyline),NaN,NaN
5,811610,MAS CAPITAL,4900145360,MANOJVID,EXW,Cstd,28.110,22.2110,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
6,811612,MAS CAPITAL,4900143920,KAVISHKARA,EXW,Cstd,17.000,45.0900,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
7,811826,UNICHELA-A053,4800474691,YasaraJ,EXW,APP,140.740,91.8500,HONG KONG,840.0,Approval,NaN,NaN
8,811827,SILUETA,4400043673,N/A,N/A,CE,14.700,5.0100,HONG KONG,NaN,Delivery term confirmation,Manual Check,Extraction Skipped for Silueta
9,811613,BODYLINE,"4502363384, 4502364384",DANUSHIW_C,EXW,APP,84.800,23.3800,HONG KONG,510.0,Financial Approval(Bodyline),NaN,NaN


In [6]:
plant_mapping

{'bodyline': {'tab_selector9': '',
  'created_by': '',
  'tab_selector1': '',
  'inco_term': ''}}

In [ ]:
import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Plant-specific mappings for selector adjustments
plant_mapping = {
    "bodyline": {
        "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9",
        "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1",
        "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }
}

# Default mapping for other plants
default_mapping = {
    "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT9",
    "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                  "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                  "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
    "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1",
    "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                 "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                 "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
}

# Function to get initial setup values
def get_initial_setup():
    print("Please provide the following initial setup values:")
    
    # Get SAPLMEGUI initial values
    sap_gui_value_plant = input("Enter the initial value for SAPLMEGUI for plant_mapping (e.g., 0013): ").strip()
    sap_gui_value_default = input("Enter the initial value for SAPLMEGUI for default_mapping (e.g., 0014): ").strip()
    
    # Get rates and weights
    rates_weights = {
        "hong_kong_exw_rate": float(input("Enter the rate for Hong Kong EXW shipments: ")),
        "hong_kong_fob_rate": float(input("Enter the rate for Hong Kong FOB shipments: ")),
        "china_exw_rate": float(input("Enter the rate for China EXW shipments: ")),
        "china_fob_rate": float(input("Enter the rate for China FOB shipments: ")),
        "max_volume_hong_kong": float(input("Enter the maximum volume for Hong Kong shipments: ")),
        "max_volume_china": float(input("Enter the maximum volume for China shipments: "))
    }
    
    return sap_gui_value_plant, sap_gui_value_default, rates_weights

# Function to replace SAPLMEGUI:XXXX with the user-provided value
def update_sap_gui_ids(sap_gui_value, mapping):
    updated_mapping = {}
    for key, value in mapping.items():
        updated_mapping[key] = value.replace("SAPLMEGUI:0013", f"SAPLMEGUI:{sap_gui_value}")
    return updated_mapping

# SAP Login Function
def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

# Function to process valid POs with SAP
def process_valid_pos_with_sap(df, session, plant_mapping, default_mapping, rates_weights):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, plant_mapping, default_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Extract PO details from SAP
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Plant-specific handling
                if plant_name in plant_mapping:
                    session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                    created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                    session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                    inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
                else:
                    # Default handling
                    session.findById(default_mapping["tab_selector9"]).select()
                    created_by = session.findById(default_mapping["created_by"]).text
                    session.findById(default_mapping["tab_selector1"]).select()
                    inco_term = session.findById(default_mapping["inco_term"]).text
                
                # Add extracted values to the sets
                merchant.add(created_by)
                incoterm.add(inco_term)
                
            except Exception as e:
                print(f"Error processing PO {purchase_order_number}: {str(e)}")
                continue

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[29] = 'Manual Check'
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
            # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[29] = 'Manual Check'
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping, default_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs)  # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
            
            # Set status
            row[28] = "Financial Approval(Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > rates_weights["max_volume_hong_kong"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > rates_weights["max_volume_china"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Received approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery term confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > rates_weights["max_volume_hong_kong"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                        row[28] = "Approval"  # Assign status
                        print(f"Hong Kong shipment, volume > {rates_weights['max_volume_hong_kong']}, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[29] = "Received approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > rates_weights["max_volume_china"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
                        row[28] = "Approval"  # Assign status
                        print(f"China shipment, volume > {rates

Logging in...
Login successful.
Navigating to transaction me23n...
Transaction opened successfully.
POs BEMIS contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
POs CHARGEURS contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
POs BEST PACIFIC  contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
Hong Kong shipment, volume > 15, Financial approval set to 'APP'.
Hong Kong shipment, volume <= 15, Incoterm mismatch. Status: 'Delivery Term Confirmation'.
POs FUJIAN UNITEX contain one

,S/O,Consignee,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks2
0,811604,BODYLINE,"4502357146, 4502359043, 4502359049, 4502359077...","Ashank, VidyaH",EXW,APP,87.740,32.8990,HONG KONG,526.440,Financial Approval(Bodyline),NaN,NaN
1,811606,UNICHELA-1900,4900143441,AVISHKAB,EXW,Cstd,74.340,21.7100,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
2,811607,BODYLINE,"4502359056, 4502365299, 4502365296, 4502365300...",VidyaH,EXW,APP,11.161,8.1830,HONG KONG,66.966,Financial Approval(Bodyline),NaN,NaN
3,811608,BODYLINE,4502333403,ISURUSIR,EXW,APP,29.000,8.3500,HONG KONG,174.000,Financial Approval(Bodyline),NaN,NaN
4,811609,BODYLINE,4502372521,ISURUSIR,EXW,APP,24.000,13.3600,HONG KONG,144.000,Financial Approval(Bodyline),NaN,NaN
5,811610,MAS CAPITAL,4900145360,MANOJVID,EXW,Cstd,28.110,22.2110,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
6,811612,MAS CAPITAL,4900143920,KAVISHKARA,EXW,Cstd,17.000,45.0900,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
7,811826,UNICHELA-A053,4800474691,YasaraJ,EXW,APP,140.740,91.8500,HONG KONG,844.440,Approval,NaN,NaN
8,811827,SILUETA,4400043673,N/A,N/A,CE,14.700,5.0100,HONG KONG,NaN,Delivery term confirmation,NaN,Extraction Skipped for Silueta
9,811613,BODYLINE,"4502363384, 4502364384",DANUSHIW_C,EXW,APP,84.800,23.3800,HONG KONG,508.800,Financial Approval(Bodyline),NaN,NaN


In [7]:
relevant_columns = [0,3, 5, 6, 7,12,13,14,17,18,28,29,30]  # SO, PO #, Merchant, Terms, Remarks
df_with_merchant_and_terms.iloc[:, relevant_columns]

,S/O,Consignee,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks2
0,811604,BODYLINE,"4502357146, 4502359043, 4502359049, 4502359077...","Ashank, VidyaH",EXW,APP,87.740,32.8990,HONG KONG,526.440,Financial Approval(Bodyline),NaN,NaN
1,811606,UNICHELA-1900,4900143441,AVISHKAB,EXW,Cstd,74.340,21.7100,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
2,811607,BODYLINE,"4502359056, 4502365299, 4502365296, 4502365300...",VidyaH,EXW,APP,11.161,8.1830,HONG KONG,66.966,Financial Approval(Bodyline),NaN,NaN
3,811608,BODYLINE,4502333403,ISURUSIR,EXW,APP,29.000,8.3500,HONG KONG,174.000,Financial Approval(Bodyline),NaN,NaN
4,811609,BODYLINE,4502372521,ISURUSIR,EXW,APP,24.000,13.3600,HONG KONG,144.000,Financial Approval(Bodyline),NaN,NaN
5,811610,MAS CAPITAL,4900145360,MANOJVID,EXW,Cstd,28.110,22.2110,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
6,811612,MAS CAPITAL,4900143920,KAVISHKARA,EXW,Cstd,17.000,45.0900,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
7,811826,UNICHELA-A053,4800474691,YasaraJ,EXW,APP,140.740,91.8500,HONG KONG,844.440,Approval,NaN,NaN
8,811827,SILUETA,4400043673,N/A,N/A,CE,14.700,5.0100,HONG KONG,NaN,Delivery term confirmation,NaN,Extraction Skipped for Silueta
9,811613,BODYLINE,"4502363384, 4502364384",DANUSHIW_C,EXW,APP,84.800,23.3800,HONG KONG,508.800,Financial Approval(Bodyline),NaN,NaN


In [9]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


# Load the workbook and the sheet
wb = load_workbook(my_file)
ws = wb[selected_sheet]

# Read the main sheet into a DataFrame
df = pd.read_excel(my_file, sheet_name=selected_sheet)

# Ensure df_with_merchant_and_terms contains only the rows to be updated
for index, row in df_with_merchant_and_terms.iterrows():
    # Find the row in the main dataframe that needs to be updated (matching on the first column, 'SO')
    matching_row = df[df.iloc[:, 0] == row.iloc[0]]  # Matching based on first column (SO)

    if not matching_row.empty:
        for col in df_with_merchant_and_terms.columns:
            if col in df.columns:
                # Find the Excel row number
                excel_row_index = df.index[df.iloc[:, 0] == row.iloc[0]].tolist()[0] + 2  # Adjust for header row
                
                # Find the Excel column letter
                excel_col_index = df.columns.get_loc(col) + 1
                excel_col_letter = get_column_letter(excel_col_index)

                # Get the value to update
                new_value = row[col]

                # Preserve date format if the column contains dates
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    ws[f"{excel_col_letter}{excel_row_index}"].value = new_value  # Update value
                    ws[f"{excel_col_letter}{excel_row_index}"].number_format = "DD-MMM-YY"  # Apply short date format
                else:
                    ws[f"{excel_col_letter}{excel_row_index}"].value = new_value  # Update non-date values

# Preserve filters (reapply if they existed)
if ws.auto_filter.ref:
    ws.auto_filter.ref = ws.auto_filter.ref

# Save the workbook (preserving formatting, date formats, and filters)
wb.save(my_file)

print("Excel file updated successfully while preserving short date format.")


Excel file updated successfully while preserving short date format.


In [15]:
relevant_columns = [0,3, 5, 6, 7,12,13,14,17,18,28,29,30]  # SO, PO #, Merchant, Terms, Remarks
df_with_merchant_and_terms.iloc[:, relevant_columns]

,SO,Column1,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks2
537,811142,MAS CAPITAL,CTC: Thinara Pathirana,NaN,"EXW GUANGDONG,CHINA",NaN,12.0,10.02,CHINA,NaN,NaN,NaN,Invalid PO
649,811746,UNICHELA-A054,4502364764,DilshanB,EXW,CE,2.1,1.67,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
650,811747,UNICHELA-A052,4502345994,DilshanB,EXW,APP,117.7,36.74,HONG KONG,706.2,Approval,NaN,NaN
651,811755,UNICHELA-A051,"4502319652, 4502337231",DILINIDE,EXW,APP,38.0,20.04,HONG KONG,228.0,Approval,NaN,NaN
652,811757,UNICHELA-A062,4900145374,suhanipe,EXW,Cstd,22.0,5.01,HONG KONG,NaN,Delivery term confirmation,NaN,NaN
653,811762,UNICHELA-A052,4502335867,KELUMW,EXW,APP,102.5,26.72,HONG KONG,615.0,Approval,NaN,NaN
654,811767,UNICHELA-A058,4800470850,DEVINDUA,EXW,APP,19.3,6.68,HONG KONG,115.8,Approval,NaN,NaN
655,811813,UNICHELA-A054,"4502358883, 4502366690, 4502367700",DilshanB,EXW,CE,4.0,3.34,HONG KONG,NaN,Delivery term confirmation,NaN,NaN


In [22]:
import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import warnings
import win32com.client
import time

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Ask the user for the SAP GUI value
sap_gui_value = input("Enter the SAP GUI value (e.g., 0013): ").strip()

# Plant-specific mappings for selector adjustments
plant_mapping = {
    "bodyline": {
        "tab_selector9": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9",
        "created_by": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1",
        "inco_term": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }
}

# Default SAP element selectors (for plants not in plant_mapping)
default_mapping = {
    "tab_selector9": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT9",
    "created_by": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                  "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                  "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
    "tab_selector1": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1",
    "inco_term": f"wnd[0]/usr/subSUB0:SAPLMEGUI:{sap_gui_value}/subSUB1:SAPLMEVIEWS:1100/"
                 "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                 "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
}



# Function to get initial setup values
def get_initial_setup():
    print("Please provide the following initial setup values:")
    
    # Get SAPLMEGUI initial value
    #sap_gui_value = input("Enter the initial value for SAPLMEGUI for bodyline (e.g., 0013): ").strip()
    
    # Get rates and weights
    rates_weights = {
        "hong_kong_exw_rate": float(input("Enter the rate for Hong Kong EXW shipments: ")),
        "hong_kong_fob_rate": float(input("Enter the rate for Hong Kong FOB shipments: ")),
        "china_exw_rate": float(input("Enter the rate for China EXW shipments: ")),
        "china_fob_rate": float(input("Enter the rate for China FOB shipments: ")),
        "max_volume_hong_kong": float(input("Enter the maximum volume for Hong Kong shipments: ")),
        "max_volume_china": float(input("Enter the maximum volume for China shipments: "))
    }
    
    return sap_gui_value, rates_weights

# Function to replace SAPLMEGUI:XXXX with the user-provided value
def update_sap_gui_ids(sap_gui_value, plant_mapping):
    for plant, mappings in plant_mapping.items():
        for key, value in mappings.items():
            mappings[key] = value.replace("SAPLMEGUI:0013", f"SAPLMEGUI:{sap_gui_value}")
    return plant_mapping

# SAP Login Function
def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

# Function to process valid POs with SAP
def process_valid_pos_with_sap(df, session, plant_mapping,default_mapping, rates_weights):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, session, plant_mapping, default_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Open PO details
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Check if plant has specific mappings, else use default
                mapping = plant_mapping.get(plant_name, default_mapping)

                # Navigate tabs and extract data
                session.findById(mapping["tab_selector9"]).select()
                created_by = session.findById(mapping["created_by"]).text
                session.findById(mapping["tab_selector1"]).select()
                inco_term = session.findById(mapping["inco_term"]).text

                # Store extracted values
                merchant.add(created_by)
                incoterm.add(inco_term)

            except Exception as e:
                print(f"Error extracting data for PO {purchase_order_number}: {e}")

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[29]='Manual Check'
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
        # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[29]='Manual Check'
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping,default_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs) # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                elif "FOB" in terms:
                    row[18] = round(max_volume * rates_weights["china_fob_rate"] / 10) * 10
            
            # Set status
            row[28] = "Financial Approval(Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > rates_weights["max_volume_hong_kong"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"] / 10) * 10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > rates_weights["max_volume_china"]:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = round(max_volume * rates_weights["china_exw_rate"] / 10) * 10
                    elif "FOB" in terms:
                        row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                    row[28] = "Financial Approval(Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery term confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Received approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery term confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > rates_weights["max_volume_hong_kong"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["hong_kong_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"Hong Kong shipment, volume > {rates_weights['max_volume_hong_kong']}, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[29] = "Received approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= {rates_weights['max_volume_hong_kong']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > rates_weights["max_volume_china"]:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = round(max_volume * rates_weights["china_exw_rate"]/10)*10
                        elif "FOB" in terms:
                            row[18] = round(max_volume * rates_weights["china_fob_rate"]/10)*10
                        row[28] = "Approval"  # Assign status
                        print(f"China shipment, volume > {rates_weights['max_volume_china']}, Financial approval set to 'APP'.")
                    
                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received approval"  # Mail status
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm match found. Status: 'All Good'. Mail to Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery term confirmation"  # Assign status for no match
                            print(f"China shipment, volume <= {rates_weights['max_volume_china']}, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

        processed_rows.append(row)

    # Create a new DataFrame with only the relevant columns
    processed_df = pd.DataFrame(processed_rows, columns=df.columns)
    return processed_df

# Main Execution
if __name__ == "__main__":
    # Get initial setup values
    sap_gui_value, rates_weights = get_initial_setup()
    
    # Update SAP GUI IDs with the user-provided value
    plant_mapping = update_sap_gui_ids(sap_gui_value, plant_mapping)
    
    # Load the Excel file and filter rows
  # Replace with your sheet name
    df = pd.read_excel(my_file, sheet_name=selected_sheet)
    
    # Filter out rows where the "status" column is blank
    df_filtered = df[df['status'].isna()]
    
    # Login to SAP
    session = sap_login(username='maheswaranl', password='Srilanka@2024', connection_name='PDM', transaction_code='me23n')
    
    # Process the filtered DataFrame
    df_with_merchant_and_terms = process_valid_pos_with_sap(df_filtered, session, plant_mapping,default_mapping, rates_weights)
    
    # Display the relevant columns
    relevant_columns = [0, 3, 5, 6, 7, 12, 13, 14, 17, 18, 28, 29, 30]  # SO, PO #, Merchant, Terms, Remarks
    (df_with_merchant_and_terms.iloc[:, relevant_columns])

Please provide the following initial setup values:
Already logged in. Navigating to transaction.
Navigating to transaction me23n...
Transaction opened successfully.


TypeError: process_valid_pos_with_sap.<locals>.extract_from_sap() missing 1 required positional argument: 'default_mapping'

In [5]:
dd

NameError: name 'dd' is not defined

In [ ]:
import datetime
from openpyxl import load_workbook
from openpyxl.styles import NamedStyle
from openpyxl.utils.cell import get_column_letter

# Define file paths
agent_file = 'New Microsoft Excel Worksheet.xlsx'  # Replace with the path to the agent's file
my_file = 'Test (1).xlsx'        # Replace with the path to your file

# Load the workbooks
agent_wb = load_workbook(agent_file, data_only=True)  # `data_only` preserves values as they appear in the cells
my_wb = load_workbook(my_file)

# Get available sheet names from the agent's workbook
agent_sheets = agent_wb.sheetnames
my_sheets = my_wb.sheetnames

# Display available sheets and ask the user to select one
print("Available sheets in the agent's workbook:")
for idx, sheet_name in enumerate(agent_sheets, 1):
    print(f"{idx}. {sheet_name}")

# Get the user's selection
while True:
    try:
        choice = int(input("Enter the number of the sheet you want to select: "))
        if 1 <= choice <= len(agent_sheets):
            selected_sheet = agent_sheets[choice - 1]
            break
        else:
            print("Invalid selection. Please select a valid sheet number.")
    except ValueError:
        print("Invalid input. Please enter a number.")

# Check if the selected sheet exists in my workbook
if selected_sheet in my_sheets:
    print(f"Sheet '{selected_sheet}' already exists in your workbook.")
    my_ws = my_wb[selected_sheet]
else:
    print(f"Sheet '{selected_sheet}' does not exist in your workbook. Creating a new sheet...")
    # Create a new sheet with the same name
    my_ws = my_wb.create_sheet(title=selected_sheet)

    # Copy headers from the previous month's sheet
    if len(my_sheets) > 0:
        # Assuming the last sheet in `my_wb` is the previous month's sheet
        previous_month_ws = my_wb[my_sheets[-1]]
        for col in range(1, previous_month_ws.max_column + 1):
            my_ws.cell(row=1, column=col).value = previous_month_ws.cell(row=1, column=col).value
        print(f"Headers copied from the previous month's sheet '{my_sheets[-1]}' to the new sheet.")
    else:
        print("No previous sheet found in your workbook to copy headers from. New sheet will be empty.")


# Select the corresponding sheet in the agent's workbook
agent_ws = agent_wb[selected_sheet]

# Identify the last non-empty row in `my_sheet.xlsx`
last_row_my_sheet = my_ws.max_row
while last_row_my_sheet > 1 and not any(my_ws.cell(row=last_row_my_sheet, column=col).value for col in range(1, my_ws.max_column + 1)):
    last_row_my_sheet -= 1  # Move up if the row is empty

# Get the SO number of the last row in your sheet
last_so_number = my_ws.cell(row=last_row_my_sheet, column=1).value

# Find the row with the last SO number in the agent sheet
agent_last_row = None
for row in range(1, agent_ws.max_row + 1):
    if agent_ws.cell(row=row, column=1).value == last_so_number:
        agent_last_row = row
        break

# Start appending from the row after the identified last SO number
start_row = (agent_last_row + 1) if agent_last_row else 2  # Skip headers if appending all rows

# Collect rows to append
rows_to_append = []
for row in range(start_row, agent_ws.max_row + 1):
    if agent_ws.cell(row=row, column=1).value:  # Ensure the first column is not empty
        rows_to_append.append([agent_ws.cell(row=row, column=col).value for col in range(1, agent_ws.max_column + 1)])

# Define a short date style
short_date_style = NamedStyle(name="short_date", number_format="MM/DD/YYYY")

# Add the style to the workbook if not already present
if "short_date" not in my_wb.named_styles:
    my_wb.add_named_style(short_date_style)

# Append the rows with proper date handling
for i, row_data in enumerate(rows_to_append, start=last_row_my_sheet + 1):
    for col, value in enumerate(row_data, start=1):
        cell = my_ws.cell(row=i, column=col, value=value)

        # Check if the value is a datetime object
        #if isinstance(value, (datetime.date, datetime.datetime)):
         #   cell.style = short_date_style  # Apply short date format

# Save the workbook
my_wb.save(my_file)
print(f"{len(rows_to_append)} new rows appended to {my_file}.")


In [47]:
# Ensure df_with_merchant_and_terms contains only the rows to be updated
# Assuming the first column (index 0) contains the unique 'SO' column
for index, row in df_with_merchant_and_terms.iterrows():
    # Find the row in the main dataframe that needs to be updated (matching on the first column, 'SO')
    matching_row = df[df.iloc[:, 0] == row.iloc[0]]  # Matching based on first column (SO)
    
    # Debugging print to check matching row(s)
    print(f"Processing SO: {row.iloc[0]}")
    #print(f"Matching row(s) in main df:\n{matching_row}")
    
    if not matching_row.empty:
        # Update the columns that need to be updated
        for col in df_with_merchant_and_terms.columns:
            col_index = df_with_merchant_and_terms.columns.get_loc(col)  # Get index of the column
            
            # Check if the column exists in the main dataframe before updating
            if col in df.columns:  
                print(f"Updating column {col} for SO = {row.iloc[0]} with value {row[col]}")
                # Update the value in the main dataframe
                df.loc[df.iloc[:, 0] == row.iloc[0], col] = row[col]
            else:
                # If the column doesn't exist in main df, print a message
                print(f"Column '{col}' not found in main dataframe!")

# Print the updated dataframe for debugging
print("\nUpdated main dataframe:")
print(df.head())

# Write the updated DataFrame back to the Excel file without affecting other rows
with pd.ExcelWriter(my_file, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    df.to_excel(writer, sheet_name=selected_sheet, index=False)

Processing SO: 811142
Updating column SO for SO = 811142 with value 811142
Updating column Booking received date for SO = 811142 with value 2025-01-20 00:00:00
Updating column Origin Air Port for SO = 811142 with value HKG
Updating column Unnamed: 3 for SO = 811142 with value MAS CAPITAL
Updating column Shipper for SO = 811142 with value FUJIAN BAIKAI
Updating column PO # for SO = 811142 with value CTC: Thinara Pathirana
Updating column Merchant for SO = 811142 with value nan
Updating column Terms for SO = 811142 with value EXW GUANGDONG,CHINA
Updating column Booked Date (MM/DD/YYYY) for SO = 811142 with value 2025-01-20 00:00:00
Updating column Product type for SO = 811142 with value LACE
Updating column Booking Quantity for SO = 811142 with value 1.0
Updating column Package Type for SO = 811142 with value ROLL
Updating column finance approved for SO = 811142 with value nan
Updating column Booking Gross KGS for SO = 811142 with value 12
Updating column Volume kgs for SO = 811142 with 

In [5]:
# Write the updated DataFrame back to the Excel file, overwriting the existing sheet
with pd.ExcelWriter(my_file, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    df.to_excel(writer, sheet_name=selected_sheet, index=False)

In [ ]:
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Assuming df_filtered is already loaded
# Verify PO numbers
def verify_po(df):
    # Check if a single PO meets criteria: starts with 4 and has 10 digits
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Validate the PO column (Column 5) for multiple POs
    def check_pos(pos):
        if pd.isna(pos):
            return "Invalid PO"

        # Split POs (assume they are comma-separated)
        po_list = str(pos).split(',')

        # Check each PO
        invalid_found = any(not is_valid_po(po.strip()) for po in po_list)
        return "Invalid PO" if invalid_found else ""

    # Apply validation to the PO # column and update Remarks (Column 30)
    df.iloc[:, 30] = df.iloc[:, 5].apply(check_pos)

    # Keep all rows for further processing and return the DataFrame
    relevant_columns = [0, 5, 30]  # SO (Column 0), PO # (Column 5), Remarks (Column 30)
    return df.iloc[:, relevant_columns]

# Apply verification and display result
df_with_po_check = verify_po(df_filtered)
(df_with_po_check)


,SO,PO #,Remarks
537,811142,CTC: Thinara Pathirana,Invalid PO
650,811747,4502345994,
651,811755,"4502319652, 4502337231",
652,811757,4900145374,
653,811762,4502335867,
654,811767,4800470850,


In [ ]:
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import win32com.client
import time# Plant-specific mappings for selector adjustments

plant_mapping = {
    "bodyline": {
        "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9",
        "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                      "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                      "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
        "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1",
        "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                     "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                     "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
    }
}
def sap_login(username, password, connection_name,transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

        # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session




def process_valid_pos_with_sap(df, session, plant_mapping):
    # Function to check if PO is valid
    def is_valid_po(po):
        return str(po).isdigit() and str(po).startswith("4") and len(str(po)) == 10

    # Function to extract Merchant and Incoterm from SAP for valid POs
    def extract_from_sap(po_list, plant_mapping):
        merchant = set()
        incoterm = set()

        for purchase_order_number, plant_name in po_list:
            try:
                # Extract PO details from SAP
                session.findById("wnd[0]/tbar[1]/btn[17]").press()
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
                session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
                
                # Plant-specific handling
                if plant_name in plant_mapping:
                    session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                    created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                    session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                    inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
                else:
                    # Default handling
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9"
                    ).select()
                    created_by = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
                    ).text
                    
                    session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1"
                    ).select()
                    inco_term = session.findById(
                        "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                        "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                        "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                    ).text
                
                # Add extracted values to the sets
                merchant.add(created_by)
                incoterm.add(inco_term)
                
            except Exception as e:
                print(f"Error processing PO {purchase_order_number}: {str(e)}")
                continue

        # Return extracted merchant and incoterm as comma-separated values
        return ", ".join(merchant), ", ".join(incoterm)

    # Process each row and validate POs
    processed_rows = []
    for index, row in df.iterrows():
        plant_name = row[3]
        po_numbers = str(row[5]).split(",")

        # Create a PO list for the current row
        po_list = [(po_number.strip(), plant_name) for po_number in po_numbers]

        # Filter valid POs
        valid_pos = [po for po in po_list if is_valid_po(po[0])]
        
        if not valid_pos:
            # Skip if no valid POs found
            row[30] = "Invalid PO"
            processed_rows.append(row)
            continue
        
        plant_name = row[3]  # 4th column
        if plant_name.lower() == "silueta":
        # Skip extraction if plant_name is silueta
            row[6] = "N/A"  # Set Merchant as "N/A"
            row[7] = "N/A"  # Set Terms as "N/A"
            row[30] = "Extraction Skipped for Silueta"
        else:
            # Extract merchant and incoterm details from SAP
            merchant, incoterm = extract_from_sap(valid_pos, plant_mapping)
            row[6] = merchant  # Merchant column

        terms = row[7]  # Terms (Incoterm) column
        shipment_type = row[17] 
        booking_gross_kgs = float(row[13])
        volume_kgs = float(row[14])
        max_volume = max(booking_gross_kgs, volume_kgs) # Assuming shipment type is in the 11th column

        if plant_name.lower() == "bodyline":
            # Set financial approval status
            row[12] = "APP"  # Assuming the 15th column is for financial approval
            
            # Calculate cost based on shipment type and terms
            if shipment_type.lower() == "hong kong":
                if "EXW" in terms:
                    row[18] = max_volume * 6  # Column 19 for "Approximate Cost ($)"
                elif "FOB" in terms:
                    row[18] = max_volume * 5.5
            elif shipment_type.lower() == "china":
                if "EXW" in terms:
                    row[18] = max_volume * 7
                elif "FOB" in terms:
                    row[18] = max_volume * 6.2
            
            # Set status
            row[28] = "Financial Approval (Bodyline)"
        
        elif plant_name.lower() == "silueta":
            # Silueta-specific logic
            if shipment_type.lower() == "hong kong":
                if max_volume > 15:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = max_volume * 6  # Column 19 for "Approximate Cost ($)"
                    elif "FOB" in terms:
                        row[18] = max_volume * 5.5
                    row[28] = "Financial Approval (Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery Term Confirmation"  # Assign status

            elif shipment_type.lower() == "china":
                if max_volume > 70:
                    row[12] = "APP"  # Assign financial approval as "APP"
                    if "EXW" in terms:
                        row[18] = max_volume * 7  # Column 19 for "Approximate Cost ($)"
                    elif "FOB" in terms:
                        row[18] = max_volume * 6.2
                    row[28] = "Financial Approval (Silueta)"  # Assign status
                else:
                    row[12] = "CE"
                    row[28] = "Delivery Term Confirmation"  # Assign status

        elif "a058" in plant_name.lower() and any(po.strip().startswith("43") for po in str(row[5]).split(",")):  
            # Assuming POs are in the 5th column and separated by commas
            # Set status for Unichela-A058 with PO starting with 43
            row[28] = "All Good"  # Assign status
            row[29] = "Received Approval"  # Assign status for mailing (assuming column 22)

        else:
            # Check if any PO starts with "49"
            if any(po.strip().startswith("49") for po in str(row[5]).split(",")):  # Column 5 for POs
                row[12] = "Cstd"  # Assign "Cstd" in the financial approval column (Column 13)
                print(f"POs {row[4]} contain one starting with '49'. Financial approval set to 'Cstd'.")

                # Check if the SAP extracted incoterm (incoterm) is in the terms column (Column 8)
                extracted_incoterm = str(row[7])  # Column 8 for terms
                print(f"Extracted incoterm from row[7]: {extracted_incoterm}, comparing with SAP incoterm: {incoterm}.")
                
                if incoterm in extracted_incoterm:
                    # If incoterm matches, set status as "All Good" and prepare mail
                    row[28] = "All Good"  # Set status as "All Good" (Column 21)
                    row[29] = "Mail to Received Approval"  # Set mail status (Column 22)
                    print(f"Incoterm {incoterm} found in extracted terms. Status set to 'All Good'. Mail to Received Approval.")
                else:
                    # If incoterm does not match, set status to "Delivery Term Confirmation"
                    row[28] = "Delivery Term Confirmation"  # Set status as "Delivery Term Confirmation" (Column 21)
                    print(f"Incoterm {incoterm} not found in extracted terms. Status set to 'Delivery Term Confirmation'.")

            else:
                if shipment_type.lower() == "hong kong":
                    if max_volume > 15:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = max_volume * 6  # Column 19 for "Approximate Cost ($)"
                        elif "FOB" in terms:
                            row[18] = max_volume * 5.5
                        row[28] = "approval"  # Assign status
                        print(f"Hong Kong shipment, volume > 15, Financial approval set to 'APP'.")

                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received Approval"  # Mail status
                            print(f"Hong Kong shipment, volume <= 15, Incoterm match found. Status: 'All Good'. Mail Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery Term Confirmation"  # Assign status for no match
                            print(f"Hong Kong shipment, volume <= 15, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                elif shipment_type.lower() == "china":
                    if max_volume > 70:
                        row[12] = "APP"  # Assign financial approval as "APP"
                        if "EXW" in terms:
                            row[18] = max_volume * 7  # Column 19 for "Approximate Cost ($)"
                        elif "FOB" in terms:
                            row[18] = max_volume * 6.2
                        row[28] = "approval"  # Assign status
                        print(f"China shipment, volume > 70, Financial approval set to 'APP'.")
                    
                    else:
                        # Check if the extracted incoterm matches and adjust accordingly
                        if incoterm in terms:
                            row[28] = "All Good"  # Status for matching incoterm
                            row[21] = "Received Approval"  # Mail status
                            print(f"China shipment, volume <= 70, Incoterm match found. Status: 'All Good'. Mail to Received Approval.")
                        else:
                            row[12] = "CE"  # Assign "CE" in financial approval
                            row[28] = "Delivery Term Confirmation"  # Assign status for no match
                            print(f"China shipment, volume <= 70, Incoterm mismatch. Status: 'Delivery Term Confirmation'.")

                


        processed_rows.append(row)

    # Create a new DataFrame with only the relevant columns
    relevant_columns = [0,3, 5, 6, 7,12,13,14,17,18,28,29,30]  # SO, PO #, Merchant, Terms, Remarks
    processed_df = pd.DataFrame(processed_rows, columns=df.columns).iloc[:, relevant_columns]

    return processed_df



session = sap_login(username='maheswaranl', password='Srilanka@2024', connection_name='PDM', transaction_code='me23n')

# Apply the function
df_with_merchant_and_terms = process_valid_pos_with_sap(df_filtered, session, plant_mapping)
(df_with_merchant_and_terms)


Already logged in. Navigating to transaction.
Navigating to transaction me23n...
Transaction opened successfully.
Hong Kong shipment, volume > 15, Financial approval set to 'APP'.
Hong Kong shipment, volume > 15, Financial approval set to 'APP'.
POs BEMIS contain one starting with '49'. Financial approval set to 'Cstd'.
Extracted incoterm from row[7]: EXW, comparing with SAP incoterm: FOB.
Incoterm FOB not found in extracted terms. Status set to 'Delivery Term Confirmation'.
Hong Kong shipment, volume > 15, Financial approval set to 'APP'.
Hong Kong shipment, volume > 15, Financial approval set to 'APP'.


,SO,Unnamed: 3,PO #,Merchant,Terms,finance approved,Booking Gross KGS,Volume kgs,Shipment Type,Approximate Cost ($),status,mail,Remarks
537,811142,MAS CAPITAL,CTC: Thinara Pathirana,NaN,"EXW GUANGDONG,CHINA",NaN,12.0,10.02,CHINA,NaN,NaN,NaN,Invalid PO
650,811747,UNICHELA-A052,4502345994,DilshanB,EXW,APP,117.7,36.74,HONG KONG,706.2,approval,NaN,
651,811755,UNICHELA-A051,"4502319652, 4502337231",DILINIDE,EXW,APP,38.0,20.04,HONG KONG,228.0,approval,NaN,
652,811757,UNICHELA-A062,4900145374,suhanipe,EXW,Cstd,22.0,5.01,HONG KONG,NaN,Delivery Term Confirmation,NaN,
653,811762,UNICHELA-A052,4502335867,KELUMW,EXW,APP,102.5,26.72,HONG KONG,615.0,approval,NaN,
654,811767,UNICHELA-A058,4800470850,DEVINDUA,EXW,APP,19.3,6.68,HONG KONG,115.8,approval,NaN,


In [9]:
def process_row(row):
    # Check consignee (shipper)
    if row[4].strip().lower() == "bodyline":  # Column index 4 is for Shipper
        row[28] = "finance approval (bodyline)"  # Set status (index 28)

    else:  # If not Bodyline, process PO numbers
        remarks = []
        finance_approval = None
        status = None
        approximate_cost = None

        # Handle multiple POs in the "PO #" column (index 5)
        po_numbers = str(row[5]).split(",")  # Assuming POs are comma-separated
        for po in po_numbers:
            po = po.strip()  # Clean spaces
            if len(po) != 10 or not po.startswith("4"):
                remarks.append("invalid po")
            elif po.startswith("49"):
                finance_approval = "Cstd"
                status = "delivery term confirmation"
            else:
                # Check max of Booking Gross KGS (13) and Volume kgs (14)
                try:
                    booking_gross_kgs = float(row[13])
                    volume_kgs = float(row[14])
                    max_value = max(booking_gross_kgs, volume_kgs)

                    # Hong Kong shipment logic
                    if row[17].strip().lower() == "hong kong":
                        if max_value > 15:
                            finance_approval = "APP"
                            if "EXW" in row[7].strip().upper():
                                approximate_cost = max_value * 6
                            elif "FOB" in row[7].strip().upper():
                                approximate_cost = max_value * 5.5
                            status = "Approval"
                        else:  # max_value <= 15
                            finance_approval = "CE"
                            status = "All Good"

                    # China shipment logic
                    elif row[17].strip().lower() == "china":
                        if max_value > 70:
                            finance_approval = "APP"
                            if "EXW" in row[7].strip().upper():
                                approximate_cost = max_value * 7
                            elif "FOB" in row[7].strip().upper():
                                approximate_cost = max_value * 6.2
                            status = "Approval"
                        else:  # max_value <= 70
                            finance_approval = "CE"
                            status = "All Good"

                except ValueError:
                    remarks.append("Invalid weight data")

        # Assign remarks, finance approval, approximate cost, and status to the row
        row[12] = finance_approval if finance_approval else row[12]  # Finance approved (index 12)
        row[18] = approximate_cost if approximate_cost else row[18]  # Approximate Cost ($) (index 18)
        row[28] = status if status else row[28]  # Status (index 28)
        row[30] = ", ".join(remarks) if remarks else row[30]  # Remarks (index 30)

    return row

# Apply the function to each row
df_filtered = df_filtered.apply(process_row, axis=1)

# Select only the relevant columns
columns_of_interest = [
    4,   # Shipper (Consignee)
    5,   # PO #
    7,   # Terms
    12,  # Finance Approved
    13,  # Booking Gross KGS
    14,  # Volume kgs
    18,  # Approximate Cost ($)
    17,  # Shipment Type
    28,  # Status
    30   # Remarks
]

df_filtered_relevant = df_filtered.iloc[:, columns_of_interest]

# Display the filtered DataFrame
(df_filtered_relevant)


C:\Users\MaheswaranL\AppData\Local\Temp\ipykernel_26012\3459586781.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[4].strip().lower() == "bodyline":  # Column index 4 is for Shipper
C:\Users\MaheswaranL\AppData\Local\Temp\ipykernel_26012\3459586781.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  po_numbers = str(row[5]).split(",")  # Assuming POs are comma-separated
C:\Users\MaheswaranL\AppData\Local\Temp\ipykernel_26012\3459586781.py:58: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataF

,Shipper,PO #,Terms,finance approved,Booking Gross KGS,Volume kgs,Approximate Cost ($),Shipment Type,status,Remarks
420,CHECKPOINT,NaN,EXW,NaN,21.850,12.5250,NaN,HONG KONG,NaN,invalid po
536,FUJIAN BAIKAI,CTC: Thinara Pathirana,"EXW GUANGDONG,CHINA",NaN,12.000,10.0200,NaN,CHINA,NaN,invalid po
634,PIONEER,4502338531,FOB,CE,11.700,6.1122,NaN,HONG KONG,All Good,NaN
635,PIONEER,"4900145743, 4900145441, 4900145484, 4900145660...",FOB,Cstd,51.100,66.8000,NaN,HONG KONG,delivery term confirmation,NaN
636,ELEMENTS,"4502351203, 4502364063, 4502364076, 4502356605",EXW,APP,70.900,3.3400,425.400,HONG KONG,Approval,NaN
637,TAI HING,4502310145,EXW,CE,5.840,1.6700,NaN,HONG KONG,All Good,NaN
638,SML,4502359627,EXW,APP,64.660,18.3700,387.960,HONG KONG,Approval,NaN
639,AVERY DENNISON,4502366686,EXW,CE,12.460,10.0200,NaN,HONG KONG,All Good,NaN
640,AVERY DENNISON,4502366685,EXW,APP,23.450,20.0400,140.700,HONG KONG,Approval,NaN
641,NEXGEN,4502366909,EXW,CE,5.200,2.7054,NaN,HONG KONG,All Good,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   SO                        645 non-null    int64         
 1   Booking received date     645 non-null    datetime64[ns]
 2   Origin Air Port           645 non-null    object        
 3   Unnamed: 3                645 non-null    object        
 4   Shipper                   645 non-null    object        
 5   PO #                      644 non-null    object        
 6   Merchant                  632 non-null    object        
 7   Terms                     644 non-null    object        
 8   Booked Date (MM/DD/YYYY)  645 non-null    datetime64[ns]
 9   Product type              645 non-null    object        
 10  Booking Quantity          644 non-null    float64       
 11  Package Type              644 non-null    object        
 12  finance approved      

In [53]:
import win32com.client
import os
import time

def sap_login(username, password, connection_name,transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

        # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

    


def extract_po_details(session, po_list, plant_mapping):
    """
    Navigates through ME23N and extracts details for a list of purchase orders.
    Handles exceptions based on Plant Name.
    """

    for purchase_order_number, plant_name in po_list:
        try:
            # Enter Purchase Order
            session.findById("wnd[0]/tbar[1]/btn[17]").press()
            session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = purchase_order_number
            session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(purchase_order_number)
            session.findById("wnd[1]/tbar[0]/btn[0]").press()
            time.sleep(2)

                    # Define plant-specific mappings for selector adjustments
            
        
            # Plant-specific handling
            if plant_name.lower() in plant_mapping:
                session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
                created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
                session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
                inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
            else:
                # Default handling
                session.findById(
                    "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                    "tabpTABHDT9"
                ).select()
                created_by = session.findById(
                    "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                    "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
                ).text

                session.findById(
                    "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                    "tabpTABHDT1"
                ).select()
                inco_term = session.findById(
                    "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                    "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                ).text

            # Print Results
            print(f"Purchase Order: {purchase_order_number}")
            #print(f"Plant Name: {plant_name}")
            print(f"Created By: {created_by}")
            print(f"Inco Term: {inco_term}")

        except Exception as e:
            print(f"Error processing Purchase Order {purchase_order_number} for Plant {plant_name}.")
            print(e)
            continue

    # Close the session
    #session.findById("wnd[0]/tbar[0]/btn[15]").press()
    print("Session closed.")


    # Replace with your actual credentials and connection name
username = "Maheswaranl"
password = "Srilanka@2024"
connection_name = "PDM"
transaction_code="me23n"

    # Define plant-specific mappings for selector adjustments
plant_mapping = {
                "bodyline": {
                    "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                                    "tabpTABHDT9",
                    "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                                "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
                    "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                                    "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                                    "tabpTABHDT1",
                    "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                                "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                }
            }
# Initialize an empty list to store PO details
po_details = []
# Log in and extract details
session = sap_login(username, password, connection_name,transaction_code)
# Extract PO details for all POs in the rowstoappend
for row_data in rows_to_append:
    po_numbers = row_data[5] if isinstance(row_data[0], list) else [row_data[5]]
    
    # Collect PO details for each PO number
    for po_number in po_numbers:
        # Assuming extract_po_details returns a dictionary or a list of dictionaries
        matching_po_details = extract_po_details(session, [(po_number, 'some_plant_name')], plant_mapping)
        
        # Add PO details to po_details
        if matching_po_details:
            po_details.append(matching_po_details)  # Adjust if extract_po_details returns multiple details

# Now that we have po_details populated, we can proceed with appending to the sheet
for i, row_data in enumerate(rows_to_append, start=last_row_my_sheet + 1):
    po_numbers = row_data[0] if isinstance(row_data[0], list) else [row_data[0]]
    
    # Initialize a list to collect the merchant names for this row
    merchant_names = []

    # Loop through each PO in the list for this row
    for po_number in po_numbers:
        matching_po_details = next((item for item in po_details if item['purchase_order'] == po_number), None)
        if matching_po_details:
            merchant_names.append(matching_po_details["merchant_name"])
        else:
            merchant_names.append("Unknown")  # Default if PO details are not found

    # Concatenate merchant names (separate them by a comma or any separator you like)
    merchant_names_str = ", ".join(merchant_names)

    # Add the merchant names to the row (assuming the "Merchant" column is the last column)
    row_data.append(merchant_names_str)

    # Append the row with the merchant name(s) to the Excel sheet
    for col, value in enumerate(row_data, start=1):
        cell = my_ws.cell(row=i, column=col, value=value)

        ## Apply date format if value is a datetime object
        #if isinstance(value, (datetime.date, datetime.datetime)):
           # cell.style = short_date_style

# Save the workbook after appending rows
my_wb.save(my_file)
print(f"{len(rows_to_append)} new rows appended to {my_file}.")


Already logged in. Navigating to transaction.
Navigating to transaction me23n...
Transaction opened successfully.
Error processing Purchase Order 4600033053 for Plant some_plant_name.
object of type 'int' has no len()
Session closed.
Error processing Purchase Order 4900142121 for Plant some_plant_name.
object of type 'int' has no len()
Session closed.
Error processing Purchase Order 4502337456 for Plant some_plant_name.
object of type 'int' has no len()
Session closed.
Error processing Purchase Order 4502355588 for Plant some_plant_name.
object of type 'int' has no len()
Session closed.
Error processing Purchase Order 4502346579, 4502346649 for Plant some_plant_name.
Property '<unknown>.text' can not be set.
Session closed.
Error processing Purchase Order 4502261393, 4502261425 for Plant some_plant_name.
Property '<unknown>.text' can not be set.
Session closed.
6 new rows appended to Test (1).xlsx.


In [ ]:
import win32com.client
import os
import time
from openpyxl import load_workbook

def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

def extract_po_details(session, po_number, plant_name, plant_mapping):
    """
    Navigates through ME23N and extracts details for a given purchase order.
    Handles exceptions based on Plant Name.
    """
    try:
        # Enter Purchase Order
        session.findById("wnd[0]/tbar[1]/btn[17]").press()
        session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = po_number
        session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(po_number)
        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(2)
        plant_mapping = {
        "bodyline": {
            "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT9",
            "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                          "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                          "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
            "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT1",
            "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
                }
            }

        # Plant-specific handling
        if plant_name.lower() in plant_mapping:
            session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
            created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
            session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
            inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
            merchant = created_by  # Assuming 'Created By' is Merchant for this case
        else:
            # Default handling
            session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT9"
            ).select()
            created_by = session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
            ).text

            session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT1"
            ).select()
            inco_term = session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
            ).text
            merchant = created_by  # Assuming 'Created By' is Merchant for this case

        return merchant, inco_term

    except Exception as e:
        print(f"Error processing Purchase Order {po_number} for Plant {plant_name}.")
        print(e)
        return None, None

def update_excel_with_merchant_terms(excel_file, plant_mapping):
    # Load the workbook
    wb = load_workbook(excel_file)
    ws = wb.active  # Assuming you're working with the active sheet, adjust if needed

    # Initialize SAP login credentials
    username = "Maheswaranl"
    password = "Srilanka@2025"
    connection_name = "PDM"
    transaction_code = "me23n"

    # Log in to SAP
    session = sap_login(username, password, connection_name, transaction_code)

    # Iterate through the rows and process each PO
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):  # Assuming data starts from row 2
        po_number = row[0].value  # Assuming PO Number is in the first column
        plant_name = row[1].value  # Assuming Plant Name is in the second column
        
        if po_number and plant_name:
            # Extract Merchant and Terms
            merchant, terms = extract_po_details(session, po_number, plant_name, plant_mapping)

            if merchant and terms:
                # Update Merchant and Terms columns (assuming they're in columns 6 and 8)
                row[5].value = merchant  # Column for Merchant
                row[7].value = terms     # Column for Terms

    # Save the updated workbook
    wb.save(excel_file)
    print(f"Excel file updated with Merchant and Terms.")

if __name__ == "__main__":
    # Define your Excel file
    excel_file = 'Test (1).xlsx'  # Replace with your actual file name

    # Define plant-specific mappings for selector adjustments
    plant_mapping = {
        "bodyline": {
            "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT9",
            "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                          "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                          "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
            "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT1",
            "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
        }
    }

    # Update the Excel sheet with Merchant and Terms
    update_excel_with_merchant_terms(excel_file, plant_mapping)

    print("Process completed.")


In [ ]:
import win32com.client
import os
import time
from openpyxl import load_workbook

def sap_login(username, password, connection_name, transaction_code):
    try:
        # Connect to SAP GUI
        SapGuiAuto = win32com.client.GetObject("SAPGUI")
        if not SapGuiAuto:
            raise Exception("SAP GUI is not running.")
        
        application = SapGuiAuto.GetScriptingEngine
        connection = application.Connections(0)  # Use existing connection if available
        session = connection.Children(0)
        
        # Check if already logged in
        if session.Info.IsLowSpeedConnection == 0:  # Example condition
            print("Already logged in. Navigating to transaction.")
        else:
            raise Exception("No active session. Please login manually.")
        
    except Exception as e:
        # If not logged in, create a new connection
        print("Logging in...")
        connection = application.OpenConnection(connection_name, True)
        session = connection.Children(0)
        
        # Enter credentials
        session.findById("wnd[0]/usr/txtRSYST-BNAME").text = username
        session.findById("wnd[0]/usr/pwdRSYST-BCODE").text = password
        session.findById("wnd[0]/tbar[0]/btn[0]").press()
    
        print("Login successful.")

    except Exception as e:
        print(f"Error during login: {str(e)}")
        exit(1)

    # Navigate to the transaction
    print(f"Navigating to transaction {transaction_code}...")
    session.findById("wnd[0]/tbar[0]/okcd").text = transaction_code
    session.findById("wnd[0]/tbar[0]/btn[0]").press()
    print("Transaction opened successfully.")
    return session

def extract_po_details(session, po_number, plant_name, plant_mapping):
    """
    Navigates through ME23N and extracts details for a given purchase order.
    Handles exceptions based on Plant Name.
    """
    try:
        # Enter Purchase Order
        session.findById("wnd[0]/tbar[1]/btn[17]").press()
        session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").text = po_number
        session.findById("wnd[1]/usr/subSUB0:SAPLMEGUI:0003/ctxtMEPO_SELECT-EBELN").caretPosition = len(po_number)
        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(2)

        # Plant-specific handling
        if plant_name.lower() in plant_mapping:
            session.findById(plant_mapping[plant_name]["tab_selector9"]).select()
            created_by = session.findById(plant_mapping[plant_name]["created_by"]).text
            session.findById(plant_mapping[plant_name]["tab_selector1"]).select()
            inco_term = session.findById(plant_mapping[plant_name]["inco_term"]).text
            merchant = created_by  # Assuming 'Created By' is Merchant for this case
        else:
            # Default handling
            session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT9"
            ).select()
            created_by = session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM"
            ).text

            session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT1"
            ).select()
            inco_term = session.findById(
                "wnd[0]/usr/subSUB0:SAPLMEGUI:0010/subSUB1:SAPLMEVIEWS:1100/"
                "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
            ).text
            merchant = created_by  # Assuming 'Created By' is Merchant for this case

        return merchant, inco_term

    except Exception as e:
        print(f"Error processing Purchase Order {po_number} for Plant {plant_name}.")
        print(e)
        return None, None

def update_excel_with_merchant_terms(excel_file, plant_mapping):
    # Load the workbook
    wb = load_workbook(excel_file)
    ws = wb.active  # Assuming you're working with the active sheet, adjust if needed

    # Initialize SAP login credentials
    username = "Maheswaranl"
    password = "Srilanka@2025"
    connection_name = "PDM"
    transaction_code = "me23n"

    # Log in to SAP
    session = sap_login(username, password, connection_name, transaction_code)

    # Iterate through the rows and process each PO
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):  # Assuming data starts from row 2
        po_number = row[0].value  # Assuming PO Number is in the first column
        plant_name = row[1].value  # Assuming Plant Name is in the second column
        
        if po_number and plant_name:
            # Extract Merchant and Terms
            merchant, terms = extract_po_details(session, po_number, plant_name, plant_mapping)

            if merchant and terms:
                # Update Merchant and Terms columns (assuming they're in columns 6 and 8)
                row[5].value = merchant  # Column for Merchant
                row[7].value = terms     # Column for Terms

    # Save the updated workbook
    wb.save(excel_file)
    print(f"Excel file updated with Merchant and Terms.")

if __name__ == "__main__":
    # Define your Excel file
    excel_file = 'Test (1).xlsx'  # Replace with your actual file name

    # Define plant-specific mappings for selector adjustments
    plant_mapping = {
        "bodyline": {
            "tab_selector9": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT9",
            "created_by": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                          "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                          "tabpTABHDT9/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1221/txtMEPO1222-EKNAM",
            "tab_selector1": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                            "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                            "tabpTABHDT1",
            "inco_term": "wnd[0]/usr/subSUB0:SAPLMEGUI:0013/subSUB1:SAPLMEVIEWS:1100/"
                         "subSUB2:SAPLMEVIEWS:1200/subSUB1:SAPLMEGUI:1102/tabsHEADER_DETAIL/"
                         "tabpTABHDT1/ssubTABSTRIPCONTROL2SUB:SAPLMEGUI:1226/ctxtMEPO1226-INCO1"
        }
    }

    # Update the Excel sheet with Merchant and Terms
    update_excel_with_merchant_terms(excel_file, plant_mapping)

    print("Process completed.")
